# 🧊 06 · 冷启动与多目标优化

> 新用户/新物品没有行为怎么办（冷启动）？点击好但成交差怎么权衡（多目标）？
> 这两块是推荐岗高阶面试题。本节给**五类冷启动方案 + ESMM/MMoE 思想与简例**。

## 📋 本节要解决的问题

| # | 问题 | 方法 |
|---|---|---|
| 1 | 用户冷启动怎么解？ | 试探/热度/内容画像/迁移 |
| 2 | 物品冷启动怎么解？ | 内容特征/类目热度/探索 |
| 3 | 点击和转化怎么一起优化？ | ESMM（多任务） |
| 4 | 多个目标怎么平衡？ | MMoE（门控专家） |

In [1]:
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Noto Sans CJK SC', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
import numpy as np
import pandas as pd
rng = np.random.default_rng(42)


## 1️⃣ 冷启动方案地图（必背）

In [2]:
print('''用户冷启动：
  1) 试探法：随机/热门榜兜底，看行为再个性化；
  2) 热度法：给全局热门（覆盖广，体验稳）；
  3) 画像法：用注册信息/授权（性别/城市/兴趣）匹配类目；
  4) 迁移法：跨平台/跨场景行为迁移（同集团账号打通）；
  5) 主动问：冷启动问卷/引导选兴趣（成本高）。

物品冷启动：
  1) 内容特征：用文本/视觉 embedding 找相似老物品；
  2) 类目热度：新物品给所在类目的平均 CTR 先验；
  3) 探索：给新物品一定曝光配额（EE 探索-利用）；
  4) 提权：新品加权/打标，扶持期观察。
''')

用户冷启动：
  1) 试探法：随机/热门榜兜底，看行为再个性化；
  2) 热度法：给全局热门（覆盖广，体验稳）；
  3) 画像法：用注册信息/授权（性别/城市/兴趣）匹配类目；
  4) 迁移法：跨平台/跨场景行为迁移（同集团账号打通）；
  5) 主动问：冷启动问卷/引导选兴趣（成本高）。

物品冷启动：
  1) 内容特征：用文本/视觉 embedding 找相似老物品；
  2) 类目热度：新物品给所在类目的平均 CTR 先验；
  3) 探索：给新物品一定曝光配额（EE 探索-利用）；
  4) 提权：新品加权/打标，扶持期观察。



## 2️⃣ 冷启动 demo：热度兜底 vs 画像匹配

In [3]:
n_item = 100
item_hot = rng.uniform(0, 1, n_item)                     # 全局热度
item_cat = rng.integers(0, 5, n_item)                    # 物品类目
user_pref = rng.integers(0, 5)                            # 新用户兴趣类目

# 热度兜底：直接按热度排
rec_hot = np.argsort(-item_hot)[:10]

# 画像匹配：先按用户兴趣类目过滤，再按热度排
cand = np.where(item_cat == user_pref)[0]
if len(cand) < 10:
    cand = np.concatenate([cand, np.where(item_cat != user_pref)[0]])
rec_img = cand[np.argsort(-item_hot[cand])][:10]

print(f'新用户兴趣类目: {user_pref}')
print('热度兜底 top5 类目:', item_cat[rec_hot[:5]])
print('画像匹配 top5 类目:', item_cat[rec_img[:5]])
assert len(rec_img) == 10
print('\n✅ 画像匹配显著提高"类目相关"（热度兜底只看全局）')

新用户兴趣类目: 1
热度兜底 top5 类目: [1 3 4 0 3]
画像匹配 top5 类目: [1 1 1 1 1]

✅ 画像匹配显著提高"类目相关"（热度兜底只看全局）


## 3️⃣ 多目标问题：CTR 与 CVR 的关联

In [4]:
# 常见痛点：点击样本多、转化样本少；直接学 CVR 受选择偏差影响
# ESMM 思路：P(转化|曝光) = P(点击|曝光) × P(转化|点击)
# 用全样本学点击率，用点击样本学"点击后转化率"，两者乘起来做整体目标
n = 100000
p_ctr = 0.10
p_cvr_given_click = 0.08
click = rng.uniform(0, 1, n) < p_ctr
convert = click & (rng.uniform(0, 1, n) < p_cvr_given_click)
print(f'点击率: {click.mean():.3f}（≈10%）')
print(f'整体转化率: {convert.mean():.3f}（≈{p_ctr*p_cvr_given_click:.1%}）')
print(f'点击后转化率: {convert[click].mean():.3f}（≈8%）')
assert abs(convert.mean() - p_ctr * p_cvr_given_click) < 0.02
print('✅ ESMM 的乘法关系演示：整体转化 = 点击 × 点击后转化')

点击率: 0.099（≈10%）
整体转化率: 0.008（≈0.8%）
点击后转化率: 0.078（≈8%）
✅ ESMM 的乘法关系演示：整体转化 = 点击 × 点击后转化


## 4️⃣ MMoE：多专家 + 门控

In [5]:
# MMoE：多个专家网络，门控为每个任务分配专家权重
# 演示：两个任务（点击/时长），两个专家
n_exp, n_tasks = 2, 2
gate = rng.uniform(0.1, 1, (n_tasks, n_exp))   # 门控权重（任务 x 专家）
gate = gate / gate.sum(axis=1, keepdims=True)
expert_out = rng.normal(0, 1, (n_exp, 1000))   # 每个专家的输出（1000 个样本）

# 每个任务 = 门控加权专家输出的组合
task_out = gate @ expert_out
print('门控权重矩阵（任务×专家）:')
print(pd.DataFrame(gate.round(3), index=['任务1:点击', '任务2:时长'], columns=['专家A', '专家B']).to_string())
print('\n任务1 输出维度:', task_out[0].shape)
print('➡️ MMoE 让不同任务"按需使用"共享专家，缓解任务冲突与负迁移')

门控权重矩阵（任务×专家）:
          专家A    专家B
任务1:点击  0.456  0.544
任务2:时长  0.465  0.535

任务1 输出维度: (1000,)
➡️ MMoE 让不同任务"按需使用"共享专家，缓解任务冲突与负迁移


## 5️⃣ 多目标面试追问（速答）

In [6]:
print('''1) 为什么不能只优化 CTR？ → 点击好但转化差，平台和商家目标不一致；
2) 多任务常见问题？       → 任务冲突、样本稀疏不均、梯度冲突（负迁移）；
3) 解决方案？            → 共享底部(硬共享)、ESMM(全样本乘法)、MMoE(门控)、PLE(多级专家)；
4) 怎么调权重？          → 业务目标换算（如 1 次成交=多少次点击）或离线搜索权重；
5) 指标怎么定义？        → 加权业务指标（GMV + 时长 + 留存 的复合目标）。
''')

1) 为什么不能只优化 CTR？ → 点击好但转化差，平台和商家目标不一致；
2) 多任务常见问题？       → 任务冲突、样本稀疏不均、梯度冲突（负迁移）；
3) 解决方案？            → 共享底部(硬共享)、ESMM(全样本乘法)、MMoE(门控)、PLE(多级专家)；
4) 怎么调权重？          → 业务目标换算（如 1 次成交=多少次点击）或离线搜索权重；
5) 指标怎么定义？        → 加权业务指标（GMV + 时长 + 留存 的复合目标）。



## 📝 自测清单

- [ ] 能背出用户冷启动五方案与物品冷启动四方案
- [ ] 能实现"热度兜底 vs 画像匹配"并说明差异
- [ ] 能解释 ESMM 乘法关系与解决的选择偏差
- [ ] 能画出 MMoE 结构（门控+专家）并说明作用
- [ ] 能回答多目标任务冲突的几种解法